In [1]:
import torch
from utils.read_data import load_data

In [2]:
PCs, labels, num_labels = load_data(
    "/home/tl855/project_pi_sk2433/shared/Hiren_2025_HiPoNet/pdo_data/", ""
)

In [3]:
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    """
    Computes multi-head attention. Supports nested or padded tensors.

    Args:
        E_q (int): Size of embedding dim for query
        E_k (int): Size of embedding dim for key
        E_v (int): Size of embedding dim for value
        E_total (int): Total embedding dim of combined heads post input projection. Each head
            has dim E_total // nheads
        nheads (int): Number of heads
        dropout (float, optional): Dropout probability. Default: 0.0
        bias (bool, optional): Whether to add bias to input projection. Default: True
    """

    def __init__(
        self,
        E_q: int,
        E_k: int,
        E_v: int,
        E_total: int,
        nheads: int,
        dropout: float = 0.0,
        bias=True,
        device=None,
        dtype=None,
    ):
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.nheads = nheads
        self.dropout = dropout
        self._qkv_same_embed_dim = E_q == E_k and E_q == E_v
        if self._qkv_same_embed_dim:
            self.packed_proj = nn.Linear(E_q, E_total * 3, bias=bias, **factory_kwargs)
        else:
            self.q_proj = nn.Linear(E_q, E_total, bias=bias, **factory_kwargs)
            self.k_proj = nn.Linear(E_k, E_total, bias=bias, **factory_kwargs)
            self.v_proj = nn.Linear(E_v, E_total, bias=bias, **factory_kwargs)
        E_out = E_q
        self.out_proj = nn.Linear(E_total, E_out, bias=bias, **factory_kwargs)
        assert E_total % nheads == 0, "Embedding dim is not divisible by nheads"
        self.E_head = E_total // nheads
        self.bias = bias

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        attn_mask=None,
        is_causal=False,
    ) -> torch.Tensor:
        """
        Forward pass; runs the following process:
            1. Apply input projection
            2. Split heads and prepare for SDPA
            3. Run SDPA
            4. Apply output projection

        Args:
            query (torch.Tensor): query of shape (``N``, ``L_q``, ``E_qk``)
            key (torch.Tensor): key of shape (``N``, ``L_kv``, ``E_qk``)
            value (torch.Tensor): value of shape (``N``, ``L_kv``, ``E_v``)
            attn_mask (torch.Tensor, optional): attention mask of shape (``N``, ``L_q``, ``L_kv``) to pass to SDPA. Default: None
            is_causal (bool, optional): Whether to apply causal mask. Default: False

        Returns:
            attn_output (torch.Tensor): output of shape (N, L_t, E_q)
        """
        # Step 1. Apply input projection
        if self._qkv_same_embed_dim:
            if query is key and key is value:
                result = self.packed_proj(query)
                query, key, value = torch.chunk(result, 3, dim=-1)
            else:
                q_weight, k_weight, v_weight = torch.chunk(
                    self.packed_proj.weight, 3, dim=0
                )
                if self.bias:
                    q_bias, k_bias, v_bias = torch.chunk(
                        self.packed_proj.bias, 3, dim=0
                    )
                else:
                    q_bias, k_bias, v_bias = None, None, None
                query, key, value = (
                    F.linear(query, q_weight, q_bias),
                    F.linear(key, k_weight, k_bias),
                    F.linear(value, v_weight, v_bias),
                )

        else:
            query = self.q_proj(query)
            key = self.k_proj(key)
            value = self.v_proj(value)

        # Step 2. Split heads and prepare for SDPA
        # reshape query, key, value to separate by head
        # (N, L_t, E_total) -> (N, L_t, nheads, E_head) -> (N, nheads, L_t, E_head)
        query = query.unflatten(-1, [self.nheads, self.E_head]).transpose(1, 2)
        # (N, L_s, E_total) -> (N, L_s, nheads, E_head) -> (N, nheads, L_s, E_head)
        key = key.unflatten(-1, [self.nheads, self.E_head]).transpose(1, 2)
        # (N, L_s, E_total) -> (N, L_s, nheads, E_head) -> (N, nheads, L_s, E_head)
        value = value.unflatten(-1, [self.nheads, self.E_head]).transpose(1, 2)

        # Step 3. Run SDPA
        # (N, nheads, L_t, E_head)
        attn_output = F.scaled_dot_product_attention(
            query, key, value, dropout_p=self.dropout, is_causal=is_causal
        )
        # (N, nheads, L_t, E_head) -> (N, L_t, nheads, E_head) -> (N, L_t, E_total)
        attn_output = attn_output.transpose(1, 2).flatten(-2)

        # Step 4. Apply output projection
        # (N, L_t, E_total) -> (N, L_t, E_out)
        attn_output = self.out_proj(attn_output)

        return attn_output

In [9]:
input_tensor = torch.nested.as_nested_tensor(PCs[:4], layout=torch.jagged, device="cpu").to_padded_tensor(0.)
F.scaled_dot_product_attention(input_tensor, input_tensor, input_tensor)

tensor([[[ 9.4668e-01,  2.0547e+00,  8.8996e-01,  ...,  2.5749e-01,
           1.1588e+00,  1.1107e+00],
         [-2.5526e-01, -3.4755e-01, -1.0231e+00,  ...,  2.0085e-01,
          -1.1120e+00, -5.7487e-01],
         [ 1.9577e-01, -1.0889e-01, -8.1096e-02,  ...,  2.9957e-01,
           6.8926e-01,  2.3476e-01],
         ...,
         [-9.8720e-08,  1.3970e-09, -4.1910e-09,  ...,  1.8626e-09,
          -9.3132e-09, -6.7987e-08],
         [-9.8720e-08,  1.3970e-09, -4.1910e-09,  ...,  1.8626e-09,
          -9.3132e-09, -6.7987e-08],
         [-9.8720e-08,  1.3970e-09, -4.1910e-09,  ...,  1.8626e-09,
          -9.3132e-09, -6.7987e-08]],

        [[ 1.3253e-01, -6.7644e-02,  8.8930e-01,  ..., -1.8126e-01,
          -4.5655e-01,  2.0290e-01],
         [-3.2939e-01, -1.4571e-01, -3.3305e-01,  ..., -4.5849e-01,
          -8.4706e-01, -7.1578e-01],
         [ 3.3464e-01,  2.8692e-01,  1.1406e+00,  ...,  2.0091e-02,
           6.8018e-01,  3.3007e-01],
         ...,
         [-2.4680e-08, -2

In [ ]:
embed_dim = PCs[0].shape[1]
layer = MultiHeadAttention(
    E_q=embed_dim, E_k=embed_dim, E_v=embed_dim, E_total=embed_dim, nheads=1
)

input_tensor = torch.nested.as_nested_tensor(PCs[:4], layout=torch.jagged)
output, _ = layer.forward(input_tensor, input_tensor, input_tensor)

W1011 17:08:38.939000 2090510 /nfs/roberts/project/pi_sk2433/tl855/PointCloudNet/.venv/lib/python3.11/site-packages/torch/nested/_internal/sdpa.py:327] Memory efficient kernel not used because:
W1011 17:08:38.941000 2090510 /nfs/roberts/project/pi_sk2433/tl855/PointCloudNet/.venv/lib/python3.11/site-packages/torch/nested/_internal/sdpa.py:330] Flash attention kernel not used because:
W1011 17:08:38.942000 2090510 /nfs/roberts/project/pi_sk2433/tl855/PointCloudNet/.venv/lib/python3.11/site-packages/torch/nested/_internal/sdpa.py:104] For NestedTensor inputs, Flash attention requires q,k,v to have the same last dimension and to be a multiple of 8 and less than or equal to 256. Got Query.size(-1): 44, Key.size(-1): 44, Value.size(-1): 44 instead.
W1011 17:08:38.943000 2090510 /nfs/roberts/project/pi_sk2433/tl855/PointCloudNet/.venv/lib/python3.11/site-packages/torch/nested/_internal/sdpa.py:333] Math attention kernel not used because:
W1011 17:08:38.944000 2090510 /nfs/roberts/project/pi_

RuntimeError: No viable backend for scaled_dot_product_attention was found.